In [11]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from ollama import chat
import os
from datasets import Dataset
from ragas import evaluate
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
from ragas.llms import llm_factory
from ragas.metrics.collections import faithfulness, answer_relevancy, context_recall
from ragas.embeddings import LangchainEmbeddingsWrapper

Настройка API ключей

In [ ]:
HF_API_KEY = ""
os.environ["HF_TOKEN"] = HF_API_KEY

In [13]:
file_path = r"/home/kagor/Загрузки/RAG/RAG_итог/RAG_инфа по распорядку и пр.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

Инициализация эмбеддингов

In [14]:
hf_embeddings_model = HuggingFaceEmbeddings(
    model_name="cointegrated/LaBSE-en-ru",
    model_kwargs={"device": "cpu"}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17908.99it/s]
BertModel LOAD REPORT from: cointegrated/LaBSE-en-ru
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Чанки и Сплиттер

In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

all_texts = []
for doc in docs:
    chunks = text_splitter.split_text(doc.page_content)
    all_texts.extend(chunks)

chunk_documents = [Document(page_content=text) for text in all_texts]

Создание или загрузка векторной базы Chroma

In [16]:
persist_directory = "./chroma_db"
collection_name = "university_docs"

if os.path.exists(persist_directory):
    vector_db = Chroma(persist_directory="./chroma_db",
                       embedding_function=hf_embeddings_model)
    results = vector_db.get()
    chunk_documents = [Document(page_content=doc) for doc in results['documents']]
else:
    vector_db = Chroma.from_documents(chunk_documents,
                                      hf_embeddings_model,
                                      persist_directory="./chroma_db")

Настройка ретриверов и поиск контекста

In [17]:
my_text = "сколько стипендия у студентов иностранцев?"

vector_retriever = vector_db.as_retriever(search_kwargs={"k": 5})

bm25_retriever = BM25Retriever.from_documents(chunk_documents)
bm25_retriever.k = 5

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.0, 1]
)

retriever_results = ensemble_retriever.invoke(my_text)

Формирование контекста из найденных чанков

In [18]:
context = "\n\n".join(
    d.page_content if hasattr(d, "page_content") else str(d)
    for d in retriever_results
)

Генерация ответа

In [19]:
response = chat(
    model='llama3.2:1b',
    messages=[
        {
            "role": "system",
            "content": (
                "Ты - помощник студентам МИФИ. "
                "Отвечай только на основе контекста. "
                "Если данных нет - напиши 'Информация отсутствует в документе'."
            )
        },
        {
            "role": "user",
            "content": (
                f"Контекст:\n{context}\n\n"
                f"Вопрос: {my_text}"
            )
        }
    ],
)

answer = response.message.content
print(answer)

Для стипендии в amount 1000 человек, не указано ни количество иностранцев, иrogenно у нас на данный момент нет информации о том, что количество иностранцев больше или меньше.

Источник информации отсутствует.
